# Importing External Data into TorchSig: Bring Your Own Data (BYOD) WAV Example
This notebook shows an example of how to import externally created data into TorchSig using a basic gnuradio generated WAV dataset with JSON metadata file format. 

This example employs a provided `WAVReader` subclass of TorchSig's `FileReader` to read a custom externally created dataset as a `StaticTorchSigDataset`.

---

In [ ]:
# ----------------------------------------------------------------------
# 1️⃣  Standard‑library imports
# ----------------------------------------------------------------------
import csv
import json
import math
import os
import sys
from pathlib import Path

# ----------------------------------------------------------------------
# 2️⃣  Third‑party imports
# ----------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as signal
import torch

# ----------------------------------------------------------------------
# 3️⃣  TorchSig (local / project‑specific) imports
# ----------------------------------------------------------------------
from torchsig.datasets.datasets import StaticTorchSigDataset
from torchsig.transforms.transforms import ComplexTo2D
from torchsig.utils.file_handlers.wav import WAVReader

sys.path.insert(0, "/usr/lib/python3/dist-packages")

In [ ]:
# Use only if gnuradio depends on numpy<2 on your system
# import sys
#!{sys.executable} -m pip install "numpy<2" --force-reinstall

## Step 1: External Data Generation Process: create synthetic data outside TorchSig workflow

If your data already exists somewhere, you can skip to Step 2.

We will write a sample dataset using gnuradio generated .wav for signal data and and csv for metadata. 

### External Synthetic Data and Metadata Generation

In [ ]:
# configuration parameters
root = "../datasets/byod_wav_example"  # data file top-level folder
seed = 1234567890  # rng seed

os.makedirs(root, exist_ok=True)  # directory for files

Below, we generate some signals (outside of TorchSig), using a gnuradio wavefile sink.

In [ ]:
# --------------------------------------------------------------
# 1) Import the class
# --------------------------------------------------------------
from examples.scripts.generate_gnuradio_examples import IQDatasetGenerator

# --------------------------------------------------------------
# 2) Create an instance
# --------------------------------------------------------------
gen = IQDatasetGenerator(
    root=root,          # where the folder hierarchy will appear
    snr_db=[-10, 0, 10],           # only three SNRs, for a smaller demo
    duration_s=0.5,                # half‑second files
    audio_rate=96_000,             # higher‑rate audio (still float32)
    seed=12345,                    # different deterministic seed
)

# --------------------------------------------------------------
# 3) Build the whole data set (this will take a few seconds)
# --------------------------------------------------------------
gen.generate()   # creates the WAVs + metadata.csv + info.json

# --------------------------------------------------------------
# 4) Quick visual sanity‑check
# --------------------------------------------------------------
gen.plot_example(mod="QPSK", snr=0)   # shows the first 2 ms + constellation

Let's verify the data that was written.

In [ ]:
example_paths = [
"BPSK/BPSK_+00dB_seed12345.wav",
"BPSK/BPSK_+10dB_seed12345.wav",
"BPSK/BPSK_-10dB_seed12345.wav",
"8PSK/8PSK_+00dB_seed12345.wav",
"8PSK/8PSK_+10dB_seed12345.wav",
"8PSK/8PSK_-10dB_seed12345.wav",
"QPSK/QPSK_+00dB_seed12345.wav",
"QPSK/QPSK_+10dB_seed12345.wav",
"QPSK/QPSK_-10dB_seed12345.wav"
]

In [ ]:
import os
from scipy.io import wavfile

wav_path = root + "/" + example_paths[0]
sr, data = wavfile.read(wav_path)          # data.shape = (N, 2)

print("Sample rate from WAV header :", sr)
print("Number of frames (samples per channel) :", data.shape[0])
print("Number of channels               :", data.shape[1])
print("Expected num_iq_samples from info.json :", 
      int(48_000 * 1.0))                 # adjust if you changed duration_s/audio_rate

In [ ]:
from scipy.io import wavfile

def quick_plot(wav_path):
    sr, data = wavfile.read(wav_path)          # int16 numpy array
    print(f'{os.path.basename(wav_path)} → {sr} Hz, {len(data)} samples')
    t = np.arange(len(data)) / sr
    x = data.astype(np.float32) / 32767.0
    plt.figure(figsize=(8,2))
    plt.plot(t[:2000], x[:2000])
    plt.title(os.path.basename(wav_path))
    plt.xlabel('Time (s)'); plt.ylabel('Amplitude')
    plt.grid(True)
    plt.show()

quick_plot(root + "/" + example_paths[0])

In [ ]:
from examples.scripts.generate_gnuradio_examples import plot_iq

example_path = os.path.abspath(
    root + "/" + example_paths[0]
)
print(f"Plotting {example_path}")
plot_iq(example_path)

## Step 2: StaticTorchSigDataset

Use `StaticTorchSigDataset` and a file handler to interface with the dataset.

In [ ]:
example_idx = 7

In [ ]:
custom_dataset = StaticTorchSigDataset(
    file_handler_class=WAVReader,
    root=root,
    transforms=[ComplexTo2D()],
    target_labels=["modcod"],
)  # transform complex data to 2D format, return custom label
print(f"Dataset size: {len(custom_dataset)}")

data, label = custom_dataset[example_idx]
print(f"Data element shape: {data.shape}")
print(f"Label: {label}")
print(data)

## Step 3: Visualize

Note: The flat-zero region at the end of the trace is normal. It is zero-padding used to ensure every record has a fixed length for batch processing. This does not indicate a signal error and can be ignored during visualization.

In [ ]:
# --------------------------------------------------------------
# Grab a sample from the dataset that returns (2, N) rows
# --------------------------------------------------------------
idx = example_idx
data, label = custom_dataset[idx]          # data = torch.Tensor (2, 1024)

print(f"Label (modcod) = {label}")
print(f"data.shape = {data.shape}")

# --------------------------------------------------------------
# Convert to NumPy and rebuild the complex IQ vector
# --------------------------------------------------------------
if isinstance(data, torch.Tensor):
    data_np = data.cpu().numpy()            # shape (2, N)
else:
    data_np = np.asarray(data)

I, Q = data_np[0], data_np[1]               # each is (N,)
cvec = I + 1j * Q                           # complex baseband, shape (N,)

# --------------------------------------------------------------
# (Optional) Re‑normalise to unit power – helps the colour scaling
# --------------------------------------------------------------
cvec = cvec / np.sqrt(np.mean(np.abs(cvec) ** 2))

# --------------------------------------------------------------
# Phase trajectory (the actual modulation)
# --------------------------------------------------------------
phase = np.angle(cvec)               # range (-π, π)
plt.figure(figsize=(10, 2))
plt.plot(phase, label="phase (rad)")
plt.title("Instantaneous phase")
plt.xlabel("Sample index")
plt.ylabel("Phase [rad]")
plt.grid(True)
plt.show()

# --------------------------------------------------------------
# IQ constellation (very useful for BPSK/QPSK)
# --------------------------------------------------------------
max_abs = max(abs(I).max(), abs(Q).max())
margin   = 0.2 * max_abs                     # 20 % extra beyond the farthest point

plt.figure(figsize=(6, 6))
plt.scatter(I, Q, s=15, alpha=0.7, edgecolors="k")
plt.title("IQ Constellation")
plt.xlabel("I (real)")
plt.ylabel("Q (imag)")

plt.xlim(-max_abs - margin, max_abs + margin)
plt.ylim(-max_abs - margin, max_abs + margin)

plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.grid(True)
plt.axis("equal")
plt.show()
